# Pipeline Cleanup Utility

Run these cells to safely wipe data, checkpoints, and tables for specific layers. 
This is useful for local development when you want to re-ingest data from the beginning (Kafka offsets).

In [95]:
from pyspark.sql import SparkSession
import shutil

# Must connect to Spark to drop tables and check active streams
spark = (
    SparkSession.builder
    .appName("CleanupUtility")
    .master("spark://spark-master:7077")
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.2.0")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")
print("Spark Session initialized.")

Spark Session initialized.


In [5]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Ecommerce Data Platform")
    .master("spark://spark-master:7077")
    .config(
        "spark.jars.packages",
        ",".join([
            "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.1",
            "io.delta:delta-spark_2.12:3.2.0"
        ])
    )
    .config(
        "spark.sql.extensions",
        "io.delta.sql.DeltaSparkSessionExtension"
    )
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog"
    )
    .getOrCreate()
)


spark.sparkContext.setLogLevel("ERROR")



In [6]:
spark

## 🛑 1. Stop Active Streams
Run this first to make sure no background queries try to write to Delta while you delete files.

In [7]:
for query in spark.streams.active:
    query.stop()
    print(f"Stopped stream: {query.name}")

print("All local notebook streams stopped.")

All local notebook streams stopped.


In [8]:
from pyspark.sql import SparkSession
import shutil  # <--- This is where it gets defined!


In [9]:
spark.streams.active

[]

## 🧹 2. Landing Layer Cleanup

In [10]:
# CRITICAL: Delete the checkpoint folder so it "forgets" where it was in Kafka
shutil.rmtree('/opt/spark-data/checkpoints/landing/raw_events', ignore_errors=True)

# Delete the Delta data and transaction logs
shutil.rmtree('/opt/spark-data/delta/landing/raw_events', ignore_errors=True)

# Drop the table from Spark's memory
spark.sql("DROP TABLE IF EXISTS landing.raw_events")

print("Cleaned up Landing layer ✅")

Cleaned up Landing layer ✅


## 🧹 3. Bronze Layer Cleanup

In [11]:
# CRITICAL: Delete the checkpoint folder so it "forgets" where it was in Landing
shutil.rmtree('/opt/spark-data/checkpoints/bronze/orders_raw', ignore_errors=True)

# Delete the Delta data and transaction logs
shutil.rmtree('/opt/spark-data/delta/bronze/orders_raw', ignore_errors=True)

# Drop the table from Spark's memory
spark.sql("DROP TABLE IF EXISTS bronze.orders_raw")

print("Cleaned up Bronze layer ✅")

Cleaned up Bronze layer ✅


In [12]:
# 1. Stop all streams
for query in spark.streams.active:
    query.stop()
# 2. Delete ALL data inside the delta and checkpoint folders
shutil.rmtree('/opt/spark-data/delta', ignore_errors=True)
shutil.rmtree('/opt/spark-data/checkpoints', ignore_errors=True)
# 3. Drop all tables by dropping the databases entirely
spark.sql("DROP DATABASE IF EXISTS landing CASCADE")
spark.sql("DROP DATABASE IF EXISTS bronze CASCADE")
print("All data, checkpoints, and tables have been completely wiped! ☢️")


All data, checkpoints, and tables have been completely wiped! ☢️
